In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Domyślne ustawienia transpilacji i opcje konfiguracji


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Abstrakcyjne Circuit muszą być transpilowane, ponieważ QPU obsługują ograniczony zestaw bramek bazowych i nie mogą wykonywać dowolnych operacji. Zadaniem Transpilera jest przekształcenie dowolnych Circuit tak, aby mogły być uruchamiane na wskazanym QPU. Odbywa się to poprzez tłumaczenie Circuit na obsługiwane bramki bazowe oraz wprowadzanie bramek SWAP w razie potrzeby, tak aby łączność Circuit odpowiadała łączności QPU.

Jak wyjaśniono w artykule [Transpilacja z użyciem menedżerów przejść](transpile-with-pass-managers), możesz utworzyć [menedżer przejść](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) za pomocą funkcji [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) i przekazać Circuit lub listę Circuit do jego metody [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run), aby je transpilować. Możesz wywołać `generate_preset_pass_manager`, podając tylko poziom optymalizacji i Backend, pozostawiając wartości domyślne dla wszystkich pozostałych opcji, lub przekazać dodatkowe argumenty do funkcji, aby precyzyjnie dostosować transpilację.

## Podstawowe użycie bez parametrów
W tym przykładzie przekazujemy Circuit i docelowe QPU do Transpilera bez podawania żadnych dodatkowych parametrów.

Utwórz Circuit i wyświetl wynik:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpiluj Circuit i wyświetl wynik:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Wszystkie dostępne parametry
Poniżej przedstawiono wszystkie dostępne parametry funkcji [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Istnieją dwie klasy argumentów: te opisujące cel kompilacji oraz te wpływające na sposób działania Transpilera.

Wszystkie parametry z wyjątkiem `optimization_level` są opcjonalne. Szczegółowe informacje znajdziesz w [dokumentacji API Transpilera](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Stopień optymalizacji Circuit. Liczba całkowita z zakresu (0 - 3). Wyższe poziomy generują lepiej zoptymalizowane Circuit, kosztem dłuższego czasu transpilacji. Więcej informacji znajdziesz w artykule [Ustawianie poziomu optymalizacji Transpilera](set-optimization).

### Parametry opisujące cel kompilacji:
Te argumenty opisują docelowe QPU do wykonania Circuit, w tym informacje takie jak mapa sprzężeń QPU (opisująca łączność Qubitów), bramki bazowe obsługiwane przez QPU oraz współczynniki błędów bramek.

Wiele z tych parametrów jest szczegółowo opisanych w artykule [Często używane parametry transpilacji](common-parameters).

<details>
  <summary>
    **Parametry QPU (`Backend`)**
  </summary>

**Parametry Backend** - Jeśli podasz `backend`, nie musisz podawać `target` ani żadnych innych opcji Backend. Podobnie, jeśli podasz `target`, nie musisz podawać `backend` ani żadnych innych opcji Backend.
- `backend` (Backend) - Jeśli jest ustawiony, Transpiler kompiluje wejściowy Circuit do tego urządzenia. Jeśli ustawiona jest jakakolwiek inna opcja wpływająca na te ustawienia, np. `coupling_map`, nadpisuje ona ustawienia z `backend`.
- `target` (Target) - Cel Transpilera Backend. Zwykle jest podawany jako część argumentu Backend, ale jeśli ręcznie skonstruowałeś obiekt Target, możesz go tutaj podać. Nadpisuje cel z `backend`.
- `backend_properties` (BackendProperties) - Właściwości zwracane przez QPU, w tym informacje o błędach bramek, błędach odczytu, czasach koherencji Qubitów itp. Aby znaleźć QPU dostarczające te informacje, uruchom `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Opcjonalne ograniczenie sprzętowe dotyczące rozdzielczości czasu instrukcji. Informacje te są dostarczane przez konfigurację QPU. Jeśli QPU nie ma żadnych ograniczeń dotyczących przydziału czasu instrukcji, `timing_constraints` wynosi `None` i nie jest wykonywane żadne dostosowanie. QPU może zgłaszać zestaw ograniczeń, mianowicie:
    - `granularity`: Wartość całkowita reprezentująca minimalną rozdzielczość bramki pulsu w jednostkach dt. Zdefiniowana przez użytkownika bramka pulsu powinna mieć czas trwania będący wielokrotnością tej wartości ziarnistości.
    - `min_length`: Wartość całkowita reprezentująca minimalną długość bramki pulsu w jednostkach dt. Zdefiniowana przez użytkownika bramka pulsu powinna być dłuższa od tej długości.
    - `pulse_alignment`: Wartość całkowita reprezentująca rozdzielczość czasu rozpoczęcia instrukcji bramki. Instrukcje bramki powinny rozpoczynać się w czasie będącym wielokrotnością tej wartości.
    - `acquire_alignment`: Wartość całkowita reprezentująca rozdzielczość czasu rozpoczęcia instrukcji pomiaru. Instrukcja pomiaru powinna rozpoczynać się w czasie będącym wielokrotnością tej wartości.
</details>

<details>
  <summary>
    **Parametry układu i topologii**
  </summary>

- `basis_gates` (List[str] | None) - Lista nazw bramek bazowych, do których należy rozwinąć Circuit. Na przykład ['u1', 'u2', 'u3', 'cx']. Jeśli `None`, nie rozwijaj.
- `coupling_map` (CouplingMap | List[List[int]]) - Kierowana mapa sprzężeń (ewentualnie niestandardowa) jako cel mapowania. Jeśli mapa sprzężeń jest symetryczna, oba kierunki muszą być określone. Obsługiwane są następujące formaty:
    - Instancja CouplingMap
    - Lista - musi być podana jako macierz sąsiedztwa, gdzie każdy wpis określa wszystkie kierunkowe interakcje dwu-Qubitowe obsługiwane przez QPU. Na przykład: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Mapowanie operacji Circuit na harmonogramy pulsów. Jeśli `None`, używana jest `instruction_schedule_map` QPU.
</details>

### Parametry wpływające na sposób działania Transpilera
Te parametry wpływają na konkretne etapy transpilacji. Niektóre z nich mogą wpływać na wiele etapów, ale dla uproszczenia zostały wymienione tylko pod jednym etapem. Jeśli podasz argument, np. `initial_layout` dla Qubitów, których chcesz użyć, ta wartość nadpisuje wszystkie przejścia, które mogłyby ją zmienić. Innymi słowy, Transpiler nie zmieni niczego, co ręcznie określisz. Szczegółowe informacje o poszczególnych etapach znajdziesz w artykule [Etapy Transpilera](transpiler-stages).

<details>
  <summary>
    **Etap inicjalizacji**
  </summary>

- `hls_config` (HLSConfig) - Opcjonalna klasa konfiguracyjna `HLSConfig`, która jest przekazywana bezpośrednio do przejścia transformacji `HighLevelSynthesis`. Ta klasa konfiguracyjna pozwala określić listy algorytmów syntezy i ich parametry dla różnych obiektów wysokiego poziomu.
- `init_method` (str) - Nazwa wtyczki do użycia w etapie inicjalizacji. Domyślnie nie jest używana zewnętrzna wtyczka. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `list_stage_plugins()` z `init` jako argumentem nazwy etapu.
- `unitary_synthesis_method` (str) - Nazwa metody syntezy unitarnej do użycia. Domyślnie używana jest metoda `default`. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Opcjonalny słownik konfiguracyjny przekazywany bezpośrednio do wtyczki syntezy unitarnej. Domyślnie to ustawienie nie ma żadnego efektu, ponieważ domyślna metoda syntezy unitarnej nie przyjmuje niestandardowej konfiguracji. Stosowanie niestandardowej konfiguracji jest konieczne tylko wtedy, gdy wtyczka syntezy unitarnej jest określona za pomocą argumentu `unitary_synthesis`. Ponieważ jest to specyficzne dla każdej wtyczki syntezy unitarnej, zapoznaj się z dokumentacją wtyczki, aby dowiedzieć się, jak używać tej opcji.
</details>

<details>
  <summary>
    **Etap układu**
  </summary>

- `initial_layout` (Layout | Dict | List) - Wstępne rozmieszczenie wirtualnych Qubitów na fizycznych Qubitach. Jeśli ten układ sprawia, że Circuit jest zgodny z ograniczeniami `coupling_map`, zostanie użyty. Końcowy układ nie jest gwarantowany jako ten sam, ponieważ Transpiler może permutować Qubity poprzez swapy lub inne środki. Szczegółowe informacje znajdziesz w [sekcji Układ początkowy.](common-parameters#initial-layout)
- `layout_method` (str) - Nazwa przejścia wyboru układu (`default`, `dense`, `sabre` i `trivial`). Może to być również nazwa zewnętrznej wtyczki do użycia w etapie układu. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `list_stage_plugins()` z `layout` jako argumentem `stage_name`. Wartość domyślna to `sabre`.
</details>

<details>
  <summary>
    **Etap routingu**
  </summary>

- `routing_method` (str) - Nazwa przejścia routingu (`basic`, `lookahead`, `default`, `sabre` lub `none`). Może to być również nazwa zewnętrznej wtyczki do użycia w etapie routingu. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `list_stage_plugins()` z `routing` jako argumentem `stage_name`. Wartość domyślna to `sabre`.
</details>

<details>
  <summary>
    **Etap tłumaczenia**
  </summary>

- `translation_method` (str) - Nazwa przejścia tłumaczenia (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Może to być również nazwa zewnętrznej wtyczki do użycia w etapie tłumaczenia. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `list_stage_plugins()` z `translation` jako argumentem `stage_name`. Wartość domyślna to `translator`.
</details>

<details>
  <summary>
    **Etap optymalizacji**
  </summary>

- `approximation_degree` (float, z zakresu 0-1 | None) - Heurystyczny regulator używany do aproksymacji Circuit (1.0 = brak aproksymacji, 0.0 = maksymalna aproksymacja). Wartość domyślna to 1.0. Podanie `None` ustawia stopień aproksymacji na zgłoszony współczynnik błędu. Więcej informacji znajdziesz w [sekcji Stopień aproksymacji](common-parameters#approx-degree).
- `optimization_method` (str) - Nazwa wtyczki do użycia w etapie optymalizacji. Domyślnie nie jest używana zewnętrzna wtyczka. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `list_stage_plugins()` z `optimization` jako argumentem `stage_name`.
</details>

<details>
  <summary>
    **Etap harmonogramowania**
  </summary>

- `scheduling_method` (str) - Nazwa przejścia harmonogramowania. Może to być również nazwa zewnętrznej wtyczki do użycia w etapie harmonogramowania. Listę zainstalowanych wtyczek możesz zobaczyć, uruchamiając `list_stage_plugins()` z `scheduling` jako argumentem `stage_name`.
  * 'as_soon_as_possible': Harmonogramuj instrukcje zachłannie, jak najwcześniej na zasobie Qubitu (alias: `asap`).
  * 'as_late_as_possible': Harmonogramuj instrukcje późno, tzn. utrzymując Qubity w stanie podstawowym, gdy to możliwe (alias: `alap`). Jest to wartość domyślna.
</details>

<details>
  <summary>
    **Wykonanie Transpilera**
  </summary>

- `seed_transpiler` (int) - Ustawia losowe ziarno dla stochastycznych części Transpilera.
</details>

Następujące wartości domyślne są używane, jeśli nie podasz żadnego z powyższych parametrów. Więcej informacji znajdziesz na [stronie referencyjnej API metody](../api/qiskit/transpiler_preset):

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>